# 07 · Checklist augmentation test (alternative Figure 6)

Implements [Figure_6_augmentation_analysis_plan.md](../federated_models/Figure_6_augmentation_analysis_plan.md) (November 7, 2026). The fixed primary analysis is **day 90, five years**: platinum primary, AVPC key secondary, NEPC exploratory. The notebook also runs the requested **day-0 and day-180 five-year analyses**, using each landmark's own frozen generalized no-DFCI checklist. Day 90 retains the original 2-of-3 definition; the additional landmarks are reported separately. Local Cox augmentations are internally validated development, not new external validation.

Run after notebook 04 has written `checklist_grid_guidelines/`. This notebook independently prepares its own input receipt; notebook 05 is not required for this figure group. Patient data, fold assignments and out-of-fold predictions stay under the protected data root. R reads only the aggregate allowlist and writes PDF/SVG figures plus aggregate source tables to **`main/alternative_06/`**, with a separate subfolder for every landmark: **`lm0_h1825/`**, **`lm90_h1825/`**, and **`lm180_h1825/`** in the configured figures repository.

The notebook first writes the reproduction, availability, missingness and cohort audits. Any unresolved reproduction discrepancy stops fitting. Missing coverage/pathogenicity/Grade Group/death inputs yield explicit unavailable sensitivities, never invented negative findings.


In [ ]:
from pathlib import Path
from dataclasses import asdict, replace
import hashlib
import json
import os
import sys

HERE = Path.cwd()
PROJECT_ROOT = next((p for p in [HERE, *HERE.parents]
                     if (p / "COMPASS" / "federated_models" / "checklist_augmentation.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run from the repository root or COMPASS/notebooks")
sys.path.insert(0, str(PROJECT_ROOT))
from COMPASS.federated_models import checklist_augmentation as ca
from COMPASS.federated_models import checklist_augmentation_inputs as inputs
from COMPASS.federated_models import federated_clinical_comparison as fcc
from survival_common import federated_endpoint_transfer as fet
import pandas as pd

DATA_ROOT = Path(os.environ.get("COMPASS_DATA_ROOT", "/data/gusev/USERS/jpconnor/data/CAIA/COMPASS"))
RHINO_ROOT = Path(os.environ.get("COMPASS_RHINO_ROOT", str(DATA_ROOT / "rhino_data_v2")))
TRANSFER_ROOT = Path(os.environ.get("COMPASS_FEDERATED_TRANSFER_NO_DFCI", str(RHINO_ROOT / "federated_transfer_adt_no_DFCI")))
OUT = Path(os.environ.get("COMPASS_CHECKLIST_AUGMENTATION_ROOT", str(DATA_ROOT / "checklist_augmentation")))
FIG_ROOT = Path(os.environ.get("COMPASS_FIG_ROOT", "/data/gusev/USERS/jpconnor/figures/CAIA/COMPASS"))
# Set this to the figures repository containing main/ and supplement/ for direct publication there.
MANUSCRIPT_FIG_ROOT = Path(os.environ.get("COMPASS_MANUSCRIPT_FIG_ROOT", str(FIG_ROOT / "ADT_figures")))
FIGURE_DATA_ROOT = Path(os.environ.get("COMPASS_FIGURE_DATA_ROOT", str(DATA_ROOT / "figure_data")))

def path_setting(name, default=None):
    value = os.environ.get(name, str(default) if default is not None else "")
    return Path(value) if value else None

PATHS = {
    "local": path_setting("COMPASS_AUGMENTATION_LOCAL", DATA_ROOT / "longitudinal_prediction_data_adt.csv"),
    "bundle": path_setting("COMPASS_AUGMENTATION_BUNDLE", RHINO_ROOT / "feature_set_models" / "no_DFCI" / "predictive_checklists" / "predictive_checklist_federated_model_adt.json"),
    "gleason": path_setting("COMPASS_AUGMENTATION_GLEASON", fcc.sg.DEFAULT_GLEASON_PATH),
    "somatic": path_setting("COMPASS_AUGMENTATION_SOMATIC", fcc.sg.DEFAULT_SOMATIC_PATH),
    "somatic_manifest": path_setting("COMPASS_AUGMENTATION_SOMATIC_MANIFEST", fcc.sg.DEFAULT_SOMATIC_MANIFEST_PATH),
    "lab_mapping": path_setting("COMPASS_AUGMENTATION_LAB_MAPPING", fet.DEFAULT_LAB_MAPPING),
    "transfer": TRANSFER_ROOT,
    "coverage": path_setting("COMPASS_AUGMENTATION_GENE_COVERAGE"),
    "death": path_setting("COMPASS_AUGMENTATION_DEATH"),
}
# Final inference reruns the entire nested fitting/tuning pipeline in 1,000 patient resamples.
# For a quick diagnostic only: COMPASS_AUGMENTATION_BOOTSTRAP_MODE=conditional,
# COMPASS_AUGMENTATION_BOOTSTRAP=200. The figure and tables label this explicitly.
CFG = ca.Settings(bootstrap=int(os.environ.get("COMPASS_AUGMENTATION_BOOTSTRAP", "1000")),
                  bootstrap_mode=os.environ.get("COMPASS_AUGMENTATION_BOOTSTRAP_MODE", "refit"))
LANDMARKS = ca.parse_landmarks(os.environ.get("COMPASS_AUGMENTATION_LANDMARKS", "90,0,180"))
CONFIGS = {lm: replace(CFG, landmark=lm) for lm in LANDMARKS}
RUN_DIRS = {lm: ca.landmark_directory(OUT, lm) for lm in LANDMARKS}
for protected in (OUT.resolve(), FIGURE_DATA_ROOT.resolve()):
    public = MANUSCRIPT_FIG_ROOT.resolve()
    if protected == public or protected in public.parents or public in protected.parents:
        raise ValueError("Patient inputs/caches and figure exports must be separate, non-nested directories")
print(f"Protected outputs: {OUT}")
print(f"R figure destination: {MANUSCRIPT_FIG_ROOT / 'main' / 'alternative_06'}")
display(pd.DataFrame([asdict(cfg) for cfg in CONFIGS.values()]))


## Input contracts and time origins

The Gleason timeline requires `DFCI_MRN`, `gleason_date`, `gleason_total`, and **`source_note_date`** for revised eligibility. A missing documentation date is unavailable; specimen date alone never establishes availability. The closest eligible observation is selected with the original rule, strictly before the landmark. Optional `gleason_primary` / `gleason_secondary` permit a Grade Group sensitivity only where they agree with the total.

The optional coverage table has one row per `person_id` (or `DFCI_MRN`) + `REPORT_DT`, with `TP53_assessable`, `RB1_assessable`, `PTEN_assessable` coded 0/1/unknown. These must attest adequate coverage for negative calls. The primary coding retains the manuscript's ANY of SNV/DEL/AMP/SV and is audited by variant class. Optional `TP53_pathogenic`, `RB1_pathogenic`, `PTEN_pathogenic` plus a nonempty `annotation_rule` give a separate, externally specified sensitivity; rules must be defined from annotations before viewing outcomes. Do not label ANY or ≥2 alterations as functional loss.

The genomic primary uses the original closest-to-ADT report + the selected landmark (0/90/180 days), excludes landmarks at/before ADT and recomputes labs, age and endpoint risk sets on this clock. The corresponding ADT-clock sensitivity selects only a report already available by that landmark. No unavailable Gleason or gene call is imputed. Age is the only routine covariate imputed, using training-fold medians. The optional verified death table requires `person_id`/`DFCI_MRN` and `DEATH_DATE`; a missing date means no recorded death in that source.

At day 0, strict pre-landmark availability means labs and Gleason must predate the anchor. The sequencing-clock day-0 extension uses the report as the prediction time and requires it to be after ADT; its anchoring report defines genomic availability. The ADT-day-0 genomic sensitivity requires a report strictly before ADT.


In [ ]:
# Preflight before archiving any completed run or writing new outputs.
inputs.require_input_files(PATHS)
FRAMES, AUDITS, DEFINITIONS = {}, {}, {}
for lm, cfg in CONFIGS.items():
    run_dir = RUN_DIRS[lm]
    print(f"\nDay {lm}; five-year horizon: {run_dir}")
    ca.start_run(run_dir, cfg)
    frames, audits, definitions = inputs.build_inputs(PATHS, cfg)
    FRAMES[lm], AUDITS[lm], DEFINITIONS[lm] = frames, audits, definitions
    config_text = json.dumps(definitions, sort_keys=True, default=str)
    config_version = hashlib.sha256(config_text.encode()).hexdigest()[:16]
    ca.write_json(run_dir / "configuration" / f"analysis_{config_version}.json", definitions)
    ca.write_json(run_dir / "analysis_config.json", definitions)
    ca.write_analysis(audits, run_dir)
    private = run_dir / "patient_level"
    private.mkdir(exist_ok=True)
    for name, table in frames.items():
        if isinstance(table, pd.DataFrame):
            table.to_csv(private / f"input_{name}.csv", index=False)
    for name in ("reproduction", "availability", "cohort_flow", "temporal_audit", "missingness", "followup_support", "annotation_audit"):
        print(name)
        display(audits[name])
    inputs.require_reproduction(audits)
    print(f"Day {lm} reproduction passed. Frozen configuration: {config_version}")


## Paired out-of-sample information test

Five outer folds are shared across models and endpoints within each cohort; original patient IDs group every bootstrap copy. Inner three-fold CV selects ridge strength by training-censoring-weighted five-year Brier score. Each fold refits age imputation, scaling and baseline hazards; pooled predictions are comparable five-year **net risks**. All fitted comparisons include age identically. Platinum and AVPC models are endpoint-specific local development. NEPC has no separately tuned local model; optional platinum-trained augmentations apply unchanged to held-out AVPC/NEPC patients.

The primary contrast is Gleason-cohort platinum `augmented − clinical`; AVPC is key secondary. Augmentation versus `age_labs` is also paired. Genomic and representation/complete-lab sensitivities are separate populations. Unsupported folds remain missing and invalidate the affected model estimate on that panel's whole risk set. No model-specific patient dropping is allowed. Bootstrap failures and valid-resample counts are saved; intervals require ≥90% of requested resamples and at least ten valid estimates.

Screening cutoffs use training score distributions to match the original flag workload, with both attainable neighboring operating points. Simple scores are prespecified and retained for the subsequent simplification assessment; no winner or clinical recommendation is selected from held-out results.

Each landmark rebuilds its own endpoint risk sets and feature windows, and is validated separately. Five years means 1,825 days after that landmark. Patient folds are shared across competing models within each run; no cross-landmark paired gain is claimed.


In [ ]:
RESULTS = {}
for lm, cfg in CONFIGS.items():
    print(f"\nDay {lm}; five-year analysis")
    results = ca.run_analyses(FRAMES[lm], cfg)
    RESULTS[lm] = results
    ca.write_analysis(results, RUN_DIRS[lm])
    # A completed landmark keeps its receipt if a subsequent landmark fails.
    ca.complete_run(RUN_DIRS[lm], cfg)
    for name in ("analysis_status", "metrics", "contrasts"):
        print(name)
        table = results[name]
        display(table[table.cohort.eq("gleason")] if "cohort" in table and name != "analysis_status" else table)


## Within-Gleason risk and screening utility

All four Gleason <8/≥8 × frozen flag negative/positive groups are retained, including empty or unsupported groups. KM gives net event risk with pointwise intervals; paired patient-bootstrap risk differences test separation within each Gleason stratum. These are not interaction claims. When verified death dates are supplied, a separate Aalen–Johansen sensitivity estimates cumulative incidence with competing death (same-day endpoint/death assigned to the endpoint). Five-year sensitivity, specificity, PPV and NPV use IPCW, never unequal-follow-up raw event fractions.


In [ ]:
for lm, results in RESULTS.items():
    print(f"Day {lm}; five-year risks and screening")
    display(results["four_groups"])
    display(results["within_gleason_differences"])
    display(results["screening"])
    if "calibration" in results:
        display(results["calibration"])


## Genomic extension and bedside simplification

Review the information test before considering the candidate rules. For a frozen M-of-N checklist, `gleason_score = L + I(Gleason ≥8)` has N+1 items, with fixed thresholds ≥M (`gleason_ge_m`) and ≥M+1 (`gleason_ge_m_plus1`). These reproduce the original four-item ≥2/≥3 rules at day 90. `gleason_tsg_score` adds `I(any qualifying TSG)` for N+2 items; `gleason_tsg_two_genes` is the ≥2-gene sensitivity. N+2-item thresholds are chosen only from training-fold workload distributions, and their neighboring operating points and fold stability are saved. Comparisons with fitted augmentation quantify simplification costs; all candidate scores remain visible irrespective of results. Availability cohorts and clocks are never pooled into a claimed routing strategy.

No minimum clinically useful AUC gain is invented here. Any decision to adopt an extension needs the study team's prespecified practical criterion and uncertainty/screening review. The figure retains the original checklist in panel a and identifies additions as candidates. There is no apparent full-data rule promoted as validation.


In [ ]:
for lm, results in RESULTS.items():
    print(f"Day {lm}; genomic and simple-score comparisons")
    metrics, contrasts = results["metrics"], results["contrasts"]
    display(metrics[metrics.cohort.str.startswith("genomic")])
    display(contrasts[contrasts.cohort.str.startswith("genomic")])
    display(metrics[metrics.model.isin(["L", "B", "gleason_score", "gleason_ge_m", "gleason_ge_m_plus1", "gleason_tsg_score", "gleason_tsg_two_genes", "augmented"])])
    if "fit_diagnostics" in results:
        fits = results["fit_diagnostics"]
        display(fits[fits.get("operating_point", pd.Series(index=fits.index, dtype=str)).notna()])
    if "bootstrap_diagnostics" in results:
        display(results["bootstrap_diagnostics"].groupby(["cohort", "status"], dropna=False).size().rename("resamples").reset_index())
    print(f"Complete receipt: {RUN_DIRS[lm] / 'run_manifest.json'}")


## R prepare and render

Run the next cell to build the R snapshot and render the four-panel alternative and separate b–d panels. The scripts are the same entry points as the rest of the manuscript. `COMPASS_FIGURE_GROUPS=checklist_augmentation` selects only this analysis. Figures and aggregate source tables go to separate `lm0_h1825/`, `lm90_h1825/`, and `lm180_h1825/` subfolders under `MANUSCRIPT_FIG_ROOT/main/alternative_06/`; patient-level data never enter those folders. The existing Figure 6 is retained.

You can also run both scripts in an R environment with the same `COMPASS_*` path settings. Rendering reuses the prepared snapshot without Python or patient sources. After new notebook results, rerun R preparation before rendering. Changes to aggregate tables without completing this notebook are rejected by checksum validation.

`COMPASS_AUGMENTATION_LANDMARKS` defaults to `90,0,180` in both Python and R. Set it to `0,180` to run only the additions, or `90` to reproduce the original single-landmark workflow. Each has its own protected tables, configuration, cache, and render receipt.


In [ ]:
import shutil
import subprocess

r_env = os.environ.copy()
r_env.update(COMPASS_DATA_ROOT=str(DATA_ROOT), COMPASS_FIG_ROOT=str(FIG_ROOT),
             COMPASS_FIGURE_DATA_ROOT=str(FIGURE_DATA_ROOT),
             COMPASS_CHECKLIST_AUGMENTATION_ROOT=str(OUT),
             COMPASS_MANUSCRIPT_FIG_ROOT=str(MANUSCRIPT_FIG_ROOT),
             COMPASS_FIGURE_GROUPS="checklist_augmentation",
             COMPASS_AUGMENTATION_LANDMARKS=",".join(map(str, LANDMARKS)))
R_SCRIPT = os.environ.get("RSCRIPT", "Rscript")
if shutil.which(R_SCRIPT) is None:
    raise FileNotFoundError("Set RSCRIPT to the Rscript executable for the figure environment")
for script in ("final_figure_data_prep.R", "run_figure_render.R"):
    command = [R_SCRIPT, str(PROJECT_ROOT / "COMPASS" / "notebooks" / script)]
    print("COMPASS_FIGURE_GROUPS=checklist_augmentation", *command)
    subprocess.run(command, cwd=PROJECT_ROOT, env=r_env, check=True)


## Interpretation and deliverables

`configuration/` holds versioned definitions and source/code provenance. `patient_level/` contains eligible input rows, item-level missingness, fold assignments and held-out predictions. `aggregate/` contains the reproduction audit, cohort/exclusion flow, follow-up and weight support, discrimination/paired contrasts, screening metrics, calibration, four-group risks, annotation/sensitivity availability, and fitting/bootstrap diagnostics. Final inference requires the 1,000-resample refit mode; conditional OOF intervals are diagnostic. Unsupported five-year horizons remain unsupported.

No replacement Results/Methods/caption claims are drafted until these real data have run and been reviewed. Preserve the original external transfer results, binary comparators and exploratory NEPC estimates in the supplement/source tables. Decision curves remain deferred until an intended action and plausible threshold range are specified.
